# Network analysis with innovation and sustainability data

**DAISY International Summer School 2026 — hands-on session (90 minutes)**
Fabrizio Fusillo, University of Turin

Patent data (OECD REGPAT), EU-funded project data (CORDIS Horizon Europe) and
publication data (OpenAlex) share one feature: the relation between actors is
never observed directly, it is *inferred* from co-participation in a document.
This session goes from those raw tables to networks, to positional measures,
and finally to the network-based indicators that end up in econometric models.

---
### Before you start (Colab only)
1. `Runtime` → `Change runtime type` → **R**
2. Run the setup cell below once (≈1 minute: it installs packages and
   downloads ~5 MB of data).
3. In VS Code / RStudio you do not need this notebook: open the `.R` scripts
   in `lesson/` and run them line by line.


## Setup — packages, data access and the three helper functions

Everything below is identical to `00_setup.R` in the repository.


In [ ]:
if (Sys.info()[["sysname"]] == "Linux") {
  codename <- tryCatch({
    os <- readLines("/etc/os-release", warn = FALSE)
    sub('.*=', '', grep("^VERSION_CODENAME=", os, value = TRUE))
  }, error = function(e) "jammy")
  if (length(codename) == 0 || codename == "") codename <- "jammy"
  options(repos = c(CRAN = sprintf(
    "https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
} else {
  options(repos = c(CRAN = "https://cloud.r-project.org"))
}
options(timeout = 1800)   # the 60s default is not enough to download bulk data

pkgs <- c("data.table",   # fast data handling (the workhorse for raw big files)
          "igraph",       # network analysis
          "Matrix",       # sparse matrices: two-mode -> one-mode projections
          "ggplot2",      # plots
          "ggraph",       # network visualisation, ggplot2 grammar
          "jsonlite")     # REST APIs (OpenAlex)
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
invisible(lapply(pkgs, library, character.only = TRUE))

## optional packages, only used in clearly marked "if you have time" chunks
## install.packages(c("sna", "intergraph", "graphlayouts"))

setDTthreads(0)           # use all available cores
set.seed(20260907)        # layouts and community detection are stochastic

## ---------------------------------------------------------------------------
## 2. Where is the data?
## ---------------------------------------------------------------------------
## The session works with small pre-processed extracts (~5 MB in total) of
## four sources. Locally they sit in ./data ; in Colab they are downloaded once
## from the course repository. Everything is read through daisy_data().
DAISY_URL <- Sys.getenv("DAISY_DATA_URL",
  "https://raw.githubusercontent.com/ffusillo/daisy-networks/main/data/")

daisy_data <- function(file) {
  local <- file.path("data", file)
  if (file.exists(local)) return(local)
  local <- file.path("lesson", "data", file)
  if (file.exists(local)) return(local)
  cache <- file.path(tempdir(), "daisy_data")
  dir.create(cache, showWarnings = FALSE, recursive = TRUE)
  dest <- file.path(cache, file)
  if (!file.exists(dest)) {
    message("downloading ", file, " ...")
    download.file(paste0(DAISY_URL, file), dest, mode = "wb", quiet = TRUE)
  }
  dest
}

## ---------------------------------------------------------------------------
## 3. HELPER 1 - from affiliation (two-mode) data to a one-mode network
## ---------------------------------------------------------------------------
## Almost all innovation network data are *indirectly observed*: we do not see
## the tie, we see two actors sharing an event (a patent, a project, a paper).
## The event x actor incidence matrix B gives the one-mode projection
##      A = t(B) %*% B
## where A[i,j] = number of events shared by actors i and j, and A[i,i] = number
## of events of actor i. Sparse matrices make this cheap even for 10^5 actors.
##
##   dt     : data.table in long format, one row = one actor in one event
##   event  : name of the event column  (patent, project, publication ...)
##   actor  : name of the actor column  (inventor, organisation, institution...)
##   max_size: drop events with more actors than this (huge events create huge
##             cliques: 1 project with 200 partners = 19,900 edges)
proj_two_mode <- function(dt, event, actor, max_size = Inf) {
  d <- unique(as.data.table(dt)[, .(ev = get(event), ac = get(actor))])
  d <- d[!is.na(ev) & !is.na(ac) & ev != "" & ac != ""]
  if (is.finite(max_size)) {
    big <- d[, .N, by = ev][N > max_size, ev]
    if (length(big)) message("dropping ", length(big), " events with > ",
                             max_size, " actors")
    d <- d[!ev %in% big]
  }
  d[, `:=`(ev = as.factor(ev), ac = as.factor(ac))]
  B <- sparseMatrix(i = as.integer(d$ev), j = as.integer(d$ac), x = 1,
                    dims = c(nlevels(d$ev), nlevels(d$ac)),
                    dimnames = list(levels(d$ev), levels(d$ac)))
  A <- Matrix::crossprod(B, B)                 # actor x actor
  n_ev <- diag(A)                              # events per actor
  diag(A) <- 0
  A <- Matrix::drop0(A)
  tri <- Matrix::summary(Matrix::triu(A))      # upper triangle -> edge list
  edges <- data.table(from = colnames(A)[tri$i],
                      to   = colnames(A)[tri$j],
                      weight = tri$x)
  list(edges = edges,
       nodes = data.table(name = colnames(A), n_events = as.numeric(n_ev)),
       incidence = B)
}

## HELPER 2 - assemble an igraph object with node attributes attached
make_net <- function(proj, node_attr = NULL, by = "name") {
  nodes <- proj$nodes
  if (!is.null(node_attr)) {
    node_attr <- copy(as.data.table(node_attr))
    node_attr[, (by) := as.character(get(by))]   # node names are always character
    node_attr <- unique(node_attr, by = by)
    nodes <- merge(nodes, node_attr, by.x = "name", by.y = by, all.x = TRUE)
  }
  graph_from_data_frame(proj$edges, directed = FALSE, vertices = nodes)
}

## HELPER 3 - the giant (largest) component, we often work on it
giant <- function(g) {
  cmp <- components(g)
  induced_subgraph(g, V(g)[cmp$membership == which.max(cmp$csize)])
}

cat("Setup complete -", R.version.string, "| igraph", as.character(packageVersion("igraph")), "\n")

---

## BLOCK 1 (~22 min) - PATENT DATA: the co-inventor network

Data: OECD REGPAT (May 2025), EPO applications, inventor-region file, joined
with CPC codes. Extract used here: all EPO patent applications with priority
year 2010-2019 that (i) have at least one Italian inventor and (ii) carry at
least one "green" CPC code (Y02/Y04S climate-change mitigation tagging).
What we do: from a raw patent-inventor table to (a) a co-invention network,
(b) inventor-level positional measures, (c) region-level indicators you can
put in a regression.


### 1. The raw data: one row = one inventor on one patent


In [ ]:
inv <- fread(daisy_data("pat_green_inventors_IT.csv.gz"))
inv
str(inv)

## appln_id  : patent application (the "event")
## person_id : REGPAT disambiguated inventor id (the "actor")
## reg_code  : NUTS-3 region of residence of the inventor, ctry_code: country
## prio_year : priority year = closest to the moment of invention

## How much data do we have?
inv[, .(patents = uniqueN(appln_id), inventors = uniqueN(person_id))]

## Patents and inventors per year
by_year <- inv[, .(patents = uniqueN(appln_id), inventors = uniqueN(person_id)),
               by = prio_year][order(prio_year)]
by_year

ggplot(by_year, aes(prio_year, patents)) +
  geom_col(fill = "grey40") +
  labs(x = NULL, y = "green EPO applications with IT inventors") +
  theme_minimal()

## Team size: this is what drives the density of the co-invention network
team <- inv[, .(size = uniqueN(person_id)), by = appln_id]
team[, .(mean = mean(size), median = as.double(median(size)), max = max(size))]
table(team$size)

## Careful: many patents have ONE inventor -> they will be isolated nodes
## Careful #2: inventors can be counted in several regions (reg_share) and
## patents in several technologies; use fractional counts when you aggregate.

## Where are the inventors? (NUTS-3 -> NUTS-2 by truncation)
inv[, nuts2 := substr(reg_code, 1, 4)]
inv[ctry_code == "IT", .(inventors = uniqueN(person_id)), by = nuts2][order(-inventors)][1:10]

### 2. From the two-mode (patent x inventor) to the co-invention network


In [ ]:
## The tie is *not observed*: we infer it from co-participation in a patent.
pr <- proj_two_mode(inv, event = "appln_id", actor = "person_id")

head(pr$edges)          # weight = number of patents co-invented
head(pr$nodes)          # n_events = number of patents of the inventor

## WARNING - the projection turns every team into a clique. The biggest patent
## in this sample has 68 inventors: that single document produces 68*67/2 =
## 2,278 ties. Whether to keep such documents is a research design choice.
team[order(-size)][1:5]
pr20 <- proj_two_mode(inv, "appln_id", "person_id", max_size = 20)
c(ties_all = nrow(pr$edges), ties_teams_below_20 = nrow(pr20$edges))

## Attach inventor attributes (one row per inventor: first region observed)
attr_inv <- inv[, .(inv_name = inv_name[1], ctry = ctry_code[1],
                    nuts2 = nuts2[1], reg = reg_code[1],
                    first_year = min(prio_year)), by = person_id]

g <- make_net(pr, node_attr = attr_inv, by = "person_id")
g

## Basic anatomy of the network
vcount(g); ecount(g)
edge_density(g)
mean(degree(g))
table(degree(g) == 0)                 # isolates = single-inventor patents only

## Components: co-invention networks are always highly fragmented
cmp <- components(g)
cmp$no                                # number of components
sort(cmp$csize, decreasing = TRUE)[1:10]
max(cmp$csize) / vcount(g)            # share of inventors in the giant component

gc_net <- giant(g)
gc_net

## Small-world-ness of the giant component
mean_distance(gc_net)
diameter(gc_net, weights = NA)
transitivity(gc_net, type = "global")   # very high: teams are cliques by construction
## benchmark against a random graph of the same size and density
rnd <- sample_gnm(vcount(gc_net), ecount(gc_net))
c(observed = transitivity(gc_net, type = "global"),
  random   = transitivity(rnd, type = "global"))

## Degree distribution: fat tailed, as usual in collaboration networks
ggplot(data.table(k = degree(g)), aes(k)) +
  geom_histogram(binwidth = 1, fill = "grey40") +
  scale_y_log10() + labs(x = "degree (number of distinct co-inventors)") +
  theme_minimal()

### 3. Positions: who matters, and in which sense?


In [ ]:
V(g)$degree      <- degree(g)
V(g)$strength    <- strength(g)                       # weighted by n. of patents
V(g)$betw        <- betweenness(g, weights = NA, normalized = TRUE)
V(g)$eigen       <- eigen_centrality(g, weights = NA)$vector
V(g)$constraint  <- constraint(g)                     # Burt: LOW = structural holes
V(g)$clust       <- transitivity(g, type = "local", isolates = "zero")

cent <- as.data.table(as_data_frame(g, what = "vertices"))
setnames(cent, "name", "person_id")

## Top inventors by different criteria - they are NOT the same people
cent[order(-degree)][1:10, .(inv_name, nuts2, n_events, degree, strength, betw)]
cent[order(-betw)][1:10,   .(inv_name, nuts2, n_events, degree, betw, constraint)]

## Are patent counts and network position the same information?
cent[n_events > 0, round(cor(.SD, use = "pairwise"), 2),
     .SDcols = c("n_events", "degree", "strength", "betw", "eigen", "constraint")]

## => centrality is correlated with productivity but far from collinear: this is
##    why network position enters innovation regressions on its own.

### 4. Visualise (only ever plot a subgraph you can actually read)


In [ ]:
sub <- giant(g)
sub <- induced_subgraph(sub, V(sub)[degree(sub) > 1])

ggraph(sub, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey75") +
  scale_edge_width(range = c(0.2, 1.5)) +
  geom_node_point(aes(size = degree, fill = ctry), shape = 21, colour = "white") +
  scale_size(range = c(1, 6)) +
  theme_graph(base_family = "sans") +
  labs(title = "Giant component, green co-invention network (IT, 2010-2019)",
       fill = "inventor country")

### 5. From nodes to variables: region-level network indicators


In [ ]:
## This is what usually ends up in an econometric model: aggregate inventor
## positions by region (or firm, or year window) and use them as regressors.
reg_ind <- cent[ctry == "IT" & nuts2 != "", .(
  inventors        = .N,
  patents          = sum(n_events),
  avg_degree       = mean(degree),
  avg_strength     = mean(strength),
  avg_betweenness  = mean(betw),
  avg_constraint   = mean(constraint, na.rm = TRUE),
  share_connected  = mean(degree > 0),
  top_inventor     = inv_name[which.max(degree)]
), by = nuts2][order(-patents)]
reg_ind[1:15]

## Cross-border openness: share of an inventor's ties that go outside the region
el <- as.data.table(as_data_frame(g, what = "edges"))
nuts_of <- setNames(V(g)$nuts2, V(g)$name)
el[, `:=`(n_from = nuts_of[from], n_to = nuts_of[to])]
ext <- rbind(el[, .(nuts2 = n_from, ext = as.integer(n_from != n_to), weight)],
             el[, .(nuts2 = n_to,   ext = as.integer(n_from != n_to), weight)])
open_reg <- ext[, .(external_tie_share = weighted.mean(ext, weight)), by = nuts2]
reg_ind <- merge(reg_ind, open_reg, by = "nuts2", all.x = TRUE)
reg_ind[order(-patents)][1:10, .(nuts2, patents, avg_degree, avg_constraint,
                                 share_connected, external_tie_share)]

fwrite(reg_ind, "output_region_network_indicators.csv")

### 6. The network boundary is a decision, not a fact


In [ ]:
## A small function that runs the whole pipeline and returns summary statistics
net_stats <- function(dt) {
  p <- proj_two_mode(dt, "appln_id", "person_id")
  gg <- make_net(p)
  cmp <- components(gg)
  data.table(inventors = vcount(gg), ties = ecount(gg),
             density = edge_density(gg),
             avg_degree = mean(degree(gg)),
             giant_share = max(cmp$csize) / vcount(gg),
             clustering = transitivity(gg, type = "global"))
}

## So far a tie exists only if two inventors share a GREEN patent. But the same
## inventors also collaborate on non-green patents. Same actors, wider boundary:
inv_all <- fread(daisy_data("pat_all_inventors_ITgreen.csv.gz"))
inv_all[, .(patents = uniqueN(appln_id), green = uniqueN(appln_id[green == 1]))]

boundary <- rbind(
  `green ties only` = net_stats(inv),
  `all co-patenting ties` = net_stats(inv_all), idcol = "boundary")
boundary
## Connectivity, average degree and the giant component all move: any statement
## about "the" position of an inventor is conditional on this choice.

## ... and so is the *population* boundary. Same pipeline, run for you on the
## full REGPAT for 17 countries (green patents, 2015-2019):
bench <- fread(daisy_data("green_coinvention_country_benchmark.csv"))
bench[order(-patents)]
## Note (i) how small the giant component is everywhere over a 5-year window,
## (ii) FI: avg_degree of 21 driven by a handful of very large teams - always
## look for the mega-document before interpreting a "dense" network.

### 7. IF WE HAVE TIME - does the network change over time?


In [ ]:
## We reuse net_stats() defined above on moving windows.
windows <- list(`2010-2013` = 2010:2013, `2014-2016` = 2014:2016,
                `2017-2019` = 2017:2019)
evo <- rbindlist(lapply(windows, function(y) net_stats(inv[prio_year %in% y])),
                 idcol = "window")
evo

## Discussion: fragmentation, densification, and the sensitivity of ALL of this
## to the length of the time window - a modelling choice, not a data property.

---

## BLOCK 2 (~20 min) - CORDIS: networks of EU-funded collaborative projects

Data: CORDIS Horizon Europe, open data (CC-BY), release 2026-08-06,
https://cordis.europa.eu/dataset  (project.csv, organization.csv,
policyPriorities.csv, euroSciVoc.csv ...)
Extract used here: the 4,604 HE projects tagged as 100% climate-relevant or
classified under a sustainability-related euroSciVoc term, and their 44,644
participations (16,116 distinct organisations).
Why this source: unlike patents and papers, here the collaboration is a
*contractual, funded and dated* relationship, with money attached to each
partner - and it covers the actors that patents miss (universities, public
bodies, NGOs, SMEs).


### 1. The data


In [ ]:
part <- fread(daisy_data("cordis_he_participants.csv.gz"))
proj <- fread(daisy_data("cordis_he_projects.csv"))

part          # one row = one organisation in one project
proj[1:3, .(project_id, acronym, start_year, programme, ec_contrib)]

part[, .(projects = uniqueN(project_id), organisations = uniqueN(org_id),
         participations = .N)]

## Consortium size: the "event size" that will drive the projection
size <- part[, .(partners = .N), by = project_id]
size[, .(mean = mean(partners), median = as.double(median(partners)), max = max(partners))]
size[order(-partners)][1:5]
proj[project_id %in% size[order(-partners)][1:3]$project_id, .(acronym, title)]

## Who participates? (HES = higher education, REC = research org, PRC = private,
## PUB = public body, OTH = other)
part[, .(participations = .N, orgs = uniqueN(org_id),
         ec_meur = round(sum(ec_contrib_org, na.rm = TRUE) / 1e6)),
     by = activity_type][order(-participations)]

## Top countries by participation and by money
part[, .(participations = .N,
         ec_meur = round(sum(ec_contrib_org, na.rm = TRUE) / 1e6)),
     by = country][order(-ec_meur)][1:15]

## Projects per year
part[, .(projects = uniqueN(project_id)), by = start_year][order(start_year)]

### 2. The organisation collaboration network


In [ ]:
## Same helper as for patents: the event is the project, the actor the partner.
## Mega-consortia (60+ partners) would dominate the topology, so we look at both.
pr_all <- proj_two_mode(part, "project_id", "org_id")
pr_lim <- proj_two_mode(part, "project_id", "org_id", max_size = 40)
c(ties_all = nrow(pr_all$edges), ties_below_40_partners = nrow(pr_lim$edges))

org_attr <- part[, .(org_name = org_name[1], country = country[1],
                     nuts = nuts[1], type = activity_type[1], sme = sme[1],
                     eur = sum(ec_contrib_org, na.rm = TRUE)), by = org_id]

g <- make_net(pr_lim, node_attr = org_attr, by = "org_id")
g

## Anatomy: EU funding networks look nothing like co-invention networks
cmp <- components(g)
c(nodes = vcount(g), edges = ecount(g), density = edge_density(g),
  components = cmp$no, giant_share = max(cmp$csize) / vcount(g),
  clustering = transitivity(g, type = "global"),
  avg_degree = mean(degree(g)))

## a connected, dense, high-clustering core: the "policy-made" network

### 3. Who is central - and does centrality mean the same as money?


In [ ]:
V(g)$degree   <- degree(g)                       # distinct partners
V(g)$strength <- strength(g)                     # partner-projects
V(g)$betw     <- betweenness(g, weights = NA, normalized = TRUE)
V(g)$eigen    <- eigen_centrality(g, weights = NA)$vector
V(g)$constr   <- constraint(g)

nodes <- as.data.table(as_data_frame(g, what = "vertices"))
nodes[order(-degree)][1:15, .(org_name, country, type, n_events, degree, betw,
                              eur_meur = round(eur / 1e6, 1))]

## Money and network position are related but not the same thing
nodes[, round(cor(cbind(n_events, degree, strength, betw, eigen, eur),
                  use = "pairwise"), 2)]

## Brokers vs hubs: rank difference tells you who bridges rather than accumulates
nodes[, `:=`(r_deg = frankv(-degree), r_betw = frankv(-betw))]
nodes[degree > 20][order(r_betw - r_deg)][1:10,
      .(org_name, country, type, degree, betw = round(betw, 4))]

### 4. Is EU research integrated, or nationally clustered?


In [ ]:
## (a) assortativity: do organisations partner with similar organisations?
assortativity_nominal(g, factor(V(g)$country))     # by country
assortativity_nominal(g, factor(V(g)$type))        # by type of organisation
assortativity_degree(g)                            # hubs with hubs?

## (b) E-I index: share of ties that cross national borders (compare with the
##     0.26-0.46 range we found for regions in the co-invention network)
el0 <- as.data.table(as_data_frame(g, what = "edges"))
ctry_of <- setNames(V(g)$country, V(g)$name)
el0[, cross := as.integer(ctry_of[from] != ctry_of[to])]
el0[, .(cross_border_share = weighted.mean(cross, weight))]

## (c) communities in the giant component and their national composition
gg <- giant(g)
comm <- cluster_louvain(gg, weights = E(gg)$weight)
length(comm); sort(sizes(comm), decreasing = TRUE)[1:8]

memb <- data.table(org_id = V(gg)$name, country = V(gg)$country,
                   type = V(gg)$type, comm = membership(comm))
top_comm <- memb[, .N, by = comm][order(-N)][1:6]$comm
## how concentrated is each community by country? (HHI = 1 -> single country)
memb[comm %in% top_comm, .(
  orgs = .N,
  top_country = names(which.max(table(country))),
  top_share = round(max(prop.table(table(country))), 2),
  hhi = round(sum(prop.table(table(country))^2), 3)), by = comm][order(-orgs)]
## benchmark: concentration of the whole network
memb[, .(hhi_all = round(sum(prop.table(table(country))^2), 3))]
## => communities are thematic-institutional, not national: the opposite of what
##    we found for co-invention. Worth a slide in any paper on EU integration.

### 5. Aggregate the same data at country level (and plot it)


In [ ]:
## The projection helper works at any level of aggregation: just change "actor".
pr_ctry <- proj_two_mode(part, "project_id", "country")
gc_ctry <- make_net(pr_ctry)
gc_ctry <- delete_edges(gc_ctry, E(gc_ctry)[weight < 30])   # keep readable
gc_ctry <- induced_subgraph(gc_ctry, V(gc_ctry)[degree(gc_ctry) > 0])

V(gc_ctry)$projects <- V(gc_ctry)$n_events
ggraph(gc_ctry, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey80", edge_alpha = .8) +
  scale_edge_width(range = c(0.1, 3)) +
  geom_node_point(aes(size = projects), fill = "#2c7fb8", shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 3, repel = TRUE) +
  scale_size(range = c(2, 12)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Country co-participation, Horizon Europe climate projects")

## Country-level indicators: raw weights favour big countries, so normalise.
## A simple revealed-collaboration index: observed ties / expected under
## independence given each country's number of participations.
el <- as.data.table(as_data_frame(gc_ctry, what = "edges"))
tot <- data.table(country = V(gc_ctry)$name, part = V(gc_ctry)$n_events)
el <- merge(merge(el, tot, by.x = "from", by.y = "country"),
            tot, by.x = "to", by.y = "country", suffixes = c("_f", "_t"))
el[, rci := weight / (part_f * part_t / sum(tot$part))]
el[order(-rci)][1:10, .(from, to, weight, rci = round(rci, 2))]
## Small neighbouring countries collaborate far above expectation: geography and
## institutional proximity survive even inside a supranational programme.

### 6. IF WE HAVE TIME - tie formation: new or repeated partners?


In [ ]:
## A dynamic indicator you can build from any project database: how much of the
## collaboration in t is with partners already met before t?
early <- proj_two_mode(part[start_year <= 2022], "project_id", "org_id")$edges
late  <- proj_two_mode(part[start_year >= 2024], "project_id", "org_id")$edges
key   <- function(d) paste(pmin(d$from, d$to), pmax(d$from, d$to))
mean(key(late) %in% key(early))       # share of repeated ties
## Repetition rate = trust/lock-in vs renewal of the consortium ecosystem.

### 7. IF WE HAVE TIME - the same data as a *topic* network


In [ ]:
## euroSciVoc classifies each project into scientific fields: projecting the
## project x field matrix gives the thematic space of EU climate research -
## exactly the logic we use for patent technology classes in block 4.
sv <- fread(daisy_data("cordis_he_scivoc.csv.gz"))
pr_topic <- proj_two_mode(sv, "project_id", "sci_voc")
gt_net <- make_net(pr_topic)
gt_net <- delete_edges(gt_net, E(gt_net)[weight < 10])
gt_net <- induced_subgraph(gt_net, V(gt_net)[degree(gt_net) > 0])
sort(degree(gt_net), decreasing = TRUE)[1:15]

ggraph(gt_net, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey85") +
  scale_edge_width(range = c(0.1, 2)) +
  geom_node_point(aes(size = n_events), fill = "#41ab5d", shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 2.6, repel = TRUE, max.overlaps = 20) +
  scale_size(range = c(1, 9)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Thematic space of Horizon Europe climate projects (euroSciVoc)")

---

## BLOCK 3 (~12 min) - PUBLICATION DATA: OpenAlex through its API

Publications get a session of their own in this school, so here we only look
at (i) how to pull relational data out of the OpenAlex API in three lines,
(ii) how the very same projection logic gives co-authorship networks, and
(iii) what changes when the actors are institutions rather than people.
OpenAlex: fully open (CC0), no key, ~250M works. Be polite: add your e-mail
("polite pool"), max 10 requests/second, 100k/day.
R packages worth knowing: openalexR (wrapper), rcrossref, europepmc,
bibliometrix. Here we use the raw API so you see what happens.


In [ ]:

MAIL <- "your.name@your.university.it"      # <- put YOUR address here
oa <- function(path, ...) {
  url <- paste0("https://api.openalex.org/", path, "&mailto=", MAIL)
  fromJSON(URLencode(url), simplifyVector = FALSE)
}

### 1. Aggregate queries: indicators without downloading any record


In [ ]:
## group_by returns counts, not records: perfect for descriptive statistics.
res <- oa(paste0("works?filter=title_and_abstract.search:circular economy,",
                 "publication_year:2015-2024&group_by=publication_year"))
by_year <- rbindlist(lapply(res$group_by, function(x)
  data.table(year = as.integer(x$key), works = x$count)))[order(year)]
by_year

## Which countries publish on it?
res <- oa(paste0("works?filter=title_and_abstract.search:circular economy,",
                 "publication_year:2020-2024&group_by=authorships.countries"))
rbindlist(lapply(res$group_by, function(x)
  data.table(country = x$key_display_name, works = x$count)))[1:15]

### 2. Record-level download: works with their authorships


In [ ]:
## 200 records per page, cursor paging. select= keeps the payload small.
fetch_works <- function(n_pages = 2) {
  q <- paste0("works?filter=title_and_abstract.search:circular economy,",
              "publication_year:2020-2024,type:article,has_orcid:true",
              "&select=id,display_name,publication_year,cited_by_count,authorships",
              "&per-page=200")
  out <- list(); cursor <- "*"
  for (i in seq_len(n_pages)) {
    r <- oa(paste0(q, "&cursor=", cursor))
    out[[i]] <- r$results; cursor <- r$meta$next_cursor
    if (is.null(cursor)) break
  }
  unlist(out, recursive = FALSE)
}

## flatten works x authors x institutions into a long table
flatten_authorships <- function(works) rbindlist(lapply(works, function(w)
  rbindlist(lapply(w$authorships, function(a) {
    ins <- if (length(a$institutions)) a$institutions else list(list())
    rbindlist(lapply(ins, function(s) data.table(
      work_id      = sub(".*/", "", w$id),
      year         = w$publication_year,
      cited_by     = w$cited_by_count,
      author_id    = sub(".*/", "", a$author$id %||% NA_character_),
      author_name  = a$author$display_name %||% NA_character_,
      inst_id      = sub(".*/", "", s$id %||% NA_character_),
      inst_name    = s$display_name %||% NA_character_,
      inst_country = s$country_code %||% NA_character_,
      inst_type    = s$type %||% NA_character_)))
  }), fill = TRUE)), fill = TRUE)

## live if the wifi cooperates, otherwise the cached extract (2,000 works)
aut <- tryCatch(flatten_authorships(fetch_works(2)),
                error = function(e) {
                  message("API unreachable, using the cached extract")
                  fread(daisy_data("openalex_ce_authorships.csv.gz"))
                })
if (nrow(aut) < 1000) aut <- fread(daisy_data("openalex_ce_authorships.csv.gz"))

aut[, .(works = uniqueN(work_id), authors = uniqueN(author_id),
        institutions = uniqueN(inst_id), countries = uniqueN(inst_country))]

### 3. Three networks out of one table (same helper as blocks 1 and 2)


In [ ]:
## (a) co-authorship between researchers
aut_attr <- unique(aut[, .(author_id, author_name, country = inst_country)],
                   by = "author_id")
g_aut <- make_net(proj_two_mode(aut, "work_id", "author_id", max_size = 30),
                  node_attr = aut_attr, by = "author_id")
cmp <- components(g_aut)
c(authors = vcount(g_aut), ties = ecount(g_aut),
  giant_share = max(cmp$csize) / vcount(g_aut))

## (b) collaboration between institutions - the level most used in economics
g_ins <- make_net(proj_two_mode(aut, "work_id", "inst_id", max_size = 30),
                  node_attr = unique(aut[!is.na(inst_id),
                                         .(inst_id, inst_name, inst_country, inst_type)]),
                  by = "inst_id")
g_ins
V(g_ins)$degree <- degree(g_ins)
V(g_ins)$betw   <- betweenness(g_ins, weights = NA, normalized = TRUE)
ins <- as.data.table(as_data_frame(g_ins, what = "vertices"))
ins[order(-degree)][1:12, .(inst_name, inst_country, inst_type,
                            papers = n_events, degree, betw = round(betw, 3))]

## (c) country co-publication network
g_ctry <- make_net(proj_two_mode(unique(aut[, .(work_id, inst_country)]),
                                 "work_id", "inst_country"))
sort(strength(g_ctry), decreasing = TRUE)[1:12]

el <- as.data.table(as_data_frame(g_ctry, what = "edges"))
el[order(-weight)][1:10]

ggraph(delete_edges(g_ctry, E(g_ctry)[weight < 5]), layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey80") +
  scale_edge_width(range = c(0.2, 3)) +
  geom_node_point(aes(size = n_events), fill = "#d95f0e", shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 3, repel = TRUE) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Country co-publication network, circular economy research")

### 4. What to keep in mind (and what connects this to the other blocks)


In [ ]:
## - Author disambiguation: OpenAlex ids are algorithmic. Same problem as
##   REGPAT person_id and CORDIS organisationID: measurement error in the NODES
##   propagates to every network statistic. Check your top-degree actors by hand.
## - Coverage/selection: our query is a keyword search; a different query is a
##   different network. Prefer topic/concept ids or a validated keyword list, and
##   always report the query in the paper.
## - Linking science and technology: patent front-page and non-patent-literature
##   citations (PATSTAT TLS214, Lens.org, Reliance-on-Science) let you build
##   *directed* science -> technology networks. That is where publication and
##   patent data meet.

---

## BLOCK 4 (~22 min) - NETWORKS AS MEASUREMENT DEVICES

("indirect" uses: when you do not study the network, you use it to build a
variable)
Idea: a knowledge base has a *co-relational* structure. Represent technologies
as nodes and their joint use as links, and the network becomes a measurement
instrument for concepts that have no direct observable counterpart:
relatedness, variety, coherence, complexity, diversification potential.
Data: OECD REGPAT green patents (Y02/Y04S), EU NUTS regions, CPC 4-digit
classes, two periods (2010-2014, 2015-2019).


### 1. The regional technology portfolios


In [ ]:
rt  <- fread(daisy_data("green_region_tech_EU.csv.gz"))
def <- fread(daisy_data("cpc4_def.csv"))              # CPC4 labels
rt

## fractional counts: a patent is split across its regions and its CPC classes,
## so that every patent contributes exactly 1 to the world total
rt[, sum(n_pat), by = period]

## NUTS-3 -> NUTS-2 (the level at which regional innovation is usually studied)
rt[, nuts2 := substr(reg_code, 1, 4)]
reg_tech <- rt[, .(n_pat = sum(n_pat)), by = .(nuts2, ctry_code, cpc4, period)]

## keep the recent period and regions/technologies with enough mass
d <- reg_tech[period == "2015-2019"]
big_reg  <- d[, .(tot = sum(n_pat)), by = nuts2][tot >= 20, nuts2]
big_tech <- d[, .(tot = sum(n_pat)), by = cpc4][tot >= 20, cpc4]
d <- d[nuts2 %in% big_reg & cpc4 %in% big_tech]
d[, .(regions = uniqueN(nuts2), technologies = uniqueN(cpc4))]

## region x technology matrix
X <- as.matrix(dcast(d, nuts2 ~ cpc4, value.var = "n_pat", fill = 0),
               rownames = "nuts2")
dim(X)

### 2. Revealed Technological Advantage: from counts to specialisation


In [ ]:
## RTA_rt = (X_rt / X_r.) / (X_.t / X_..)   ("Balassa index")
RTA <- (X / rowSums(X)) / rep(colSums(X) / sum(X), each = nrow(X))
M   <- (RTA >= 1) * 1                        # binary specialisation matrix
mean(M)                                      # density of the two-mode network

diversity <- rowSums(M)                      # n. of technologies of a region
ubiquity  <- colSums(M)                      # n. of regions with that technology
sort(diversity, decreasing = TRUE)[1:10]
merge(data.table(cpc4 = names(ubiquity), ubiquity), def,
      by = "cpc4")[order(ubiquity)][1:8]                # most exclusive classes

### 3. Two ways of measuring RELATEDNESS between technologies


In [ ]:
## (A) co-classification inside patents (the "knowledge space" proper):
##     two classes are related if inventors combine them in the same document
cooc <- fread(daisy_data("green_tech_cooc_EU.csv.gz"))
npat <- fread(daisy_data("green_tech_npat_EU.csv"))
techs <- colnames(X)
cooc  <- cooc[cpc4_i %in% techs & cpc4_j %in% techs]
C <- sparseMatrix(i = match(cooc$cpc4_i, techs), j = match(cooc$cpc4_j, techs),
                  x = cooc$n_cooc, dims = c(length(techs), length(techs)),
                  dimnames = list(techs, techs), symmetric = FALSE)
C <- as.matrix(C + t(C))
n_t <- setNames(npat$n_pat, npat$cpc4)[techs]
## association strength (Van Eck & Waltman): observed / expected co-occurrence
Phi_pat <- C / outer(n_t, n_t) * sum(n_t) / 2
Phi_pat[!is.finite(Phi_pat)] <- 0

## (B) co-specialisation across regions (Hidalgo et al. "proximity"):
##     two technologies are related if the same regions are good at both
Co <- t(M) %*% M
Phi_reg <- Co / outer(ubiquity, ubiquity, pmax)      # min conditional probability
Phi_reg[!is.finite(Phi_reg)] <- 0
diag(Phi_reg) <- 0

## Do the two measures agree? (they answer different questions!)
iu <- upper.tri(Phi_reg)
cor(Phi_pat[iu], Phi_reg[iu], method = "spearman")

### 4. The green knowledge space, drawn


In [ ]:
## keep the strongest links only, otherwise the map is a hairball
thr <- quantile(Phi_pat[iu], 0.98)
A <- Phi_pat * (Phi_pat >= thr)
g_ks <- graph_from_adjacency_matrix(A, mode = "undirected", weighted = TRUE, diag = FALSE)
g_ks <- induced_subgraph(g_ks, V(g_ks)[degree(g_ks) > 0])
V(g_ks)$patents <- n_t[V(g_ks)$name]
V(g_ks)$label   <- def$label[match(V(g_ks)$name, def$cpc4)]
V(g_ks)$comm    <- membership(cluster_louvain(g_ks, weights = E(g_ks)$weight))
g_ks

## which technologies bridge the green knowledge space? (candidate GPTs)
sort(betweenness(g_ks, weights = NA), decreasing = TRUE)[1:10]

ggraph(g_ks, layout = "stress") +
  geom_edge_link0(aes(edge_width = weight), edge_colour = "grey85") +
  scale_edge_width(range = c(0.1, 1.5)) +
  geom_node_point(aes(size = patents, fill = factor(comm)), shape = 21, colour = "white") +
  geom_node_text(aes(label = name), size = 2.4, repel = TRUE, max.overlaps = 30) +
  scale_size(range = c(1, 10)) +
  theme_graph(base_family = "sans") + theme(legend.position = "none") +
  labs(title = "Green knowledge space (CPC4 co-classification, EU 2015-2019)")

### 5. Region-level indicators built ON the network


In [ ]:
sh <- X / rowSums(X)                  # patent shares of each region

## (a) VARIETY = entropy of the portfolio, decomposed into related (within
##     3-digit CPC groups) and unrelated (between groups) variety
grp <- substr(colnames(X), 1, 3)
H <- function(p) { p <- p[p > 0]; -sum(p * log2(p)) }
variety <- apply(sh, 1, H)
grp_sh  <- t(rowsum(t(sh), grp))                      # shares by 3-digit group
unrelated <- apply(grp_sh, 1, H)                      # between-group entropy
related   <- variety - unrelated                      # within-group entropy

## (b) COHERENCE = average relatedness of the technologies a region holds,
##     weighted by their importance in the portfolio (Nesta & Saviotti)
coherence <- as.numeric(rowSums((sh %*% Phi_pat) * sh))

## (c) RELATEDNESS DENSITY of the technologies the region is NOT (yet) in:
##     how "close" is a new technology to what the region already does?
dens <- (M %*% Phi_reg) / rep(colSums(Phi_reg), each = nrow(M)) * 100
avg_density_out <- rowSums(dens * (1 - M)) / rowSums(1 - M)
## CAVEAT: averaged over a region, relatedness density is almost collinear with
## diversity (see the correlation matrix below). The informative variation is at
## the region x technology level - which is exactly how it is used in section 6.

## (d) COMPLEXITY of the regional portfolio
##     Hidalgo & Hausmann (2009) - applied to technologies by Balland & Rigby
##     (2017). Two equivalent readings of the same bipartite network:
##
##  d.1 METHOD OF REFLECTIONS - iterate "average of my neighbours' average"
reflections <- function(M, iter = 2) {
  kr <- rowSums(M); kt <- colSums(M)
  for (i in seq_len(iter)) {
    kr_new <- as.numeric((M %*% kt) / rowSums(M))
    kt_new <- as.numeric((t(M) %*% kr) / colSums(M))
    kr <- kr_new; kt <- kt_new
  }
  list(kr = setNames(kr, rownames(M)), kt = setNames(kt, colnames(M)))
}
## careful with the parity of the iteration: odd orders measure average
## UBIQUITY (high = simple), even orders average DIVERSITY (high = complex)
r1 <- reflections(M, 1); r2 <- reflections(M, 2)
c(order1_vs_diversity = cor(r1$kr, diversity),
  order2_vs_diversity = cor(r2$kr, diversity))

##  d.2 EIGENVECTOR FORM (what the Atlas of Economic Complexity computes)
##      Mtilde = D^-1 M U^-1 M' ; complexity = 2nd eigenvector, standardised
eci_eigen <- function(M) {
  d <- rowSums(M); u <- colSums(M)
  keep_r <- d > 0; keep_c <- u > 0
  Ms <- M[keep_r, keep_c, drop = FALSE]
  d <- rowSums(Ms); u <- colSums(Ms)
  Mt <- (Ms / d) %*% t(sweep(Ms, 2, u, "/"))   # region x region
  ev <- eigen(Mt)
  k  <- Re(ev$vectors[, 2])                 # 2nd eigenvector
  k  <- as.numeric(scale(k))
  if (cor(k, d) < 0) k <- -k                # sign convention: diversified = complex
  setNames(k, rownames(Ms))
}
kci <- eci_eigen(M)
tci <- eci_eigen(t(M))                      # same object, transposed: tech complexity
c(kci_vs_reflections = cor(kci[names(r2$kr)], r2$kr))
## the two implementations agree only loosely in small samples: report which one
## you use (the eigenvector form is the current standard)

## most and least complex green technologies
merge(data.table(cpc4 = names(tci), tci), def, by = "cpc4")[order(-tci)][1:6]
merge(data.table(cpc4 = names(tci), tci), def, by = "cpc4")[order(tci)][1:6]

## put everything together: one row per region, ready for a regression
ind <- data.table(nuts2 = rownames(X),
                  patents = rowSums(X),
                  diversity, variety, related_variety = related,
                  unrelated_variety = unrelated,
                  coherence, relatedness_density = avg_density_out,
                  complexity = kci[rownames(X)])
ind[, country := substr(nuts2, 1, 2)]
ind[order(-complexity)][1:12]
round(cor(ind[, .(patents, diversity, variety, related_variety,
                  unrelated_variety, coherence, relatedness_density, complexity)]), 2)

fwrite(ind, "output_region_knowledge_indicators.csv")

ggplot(ind, aes(log(patents), complexity, label = nuts2)) +
  geom_point(aes(size = variety), alpha = .6, colour = "#2c7fb8") +
  geom_text(size = 2.4, vjust = -1, check_overlap = TRUE) +
  labs(x = "log green patents", y = "complexity of the portfolio (KCI)",
       size = "variety") + theme_minimal()

### 6. IF WE HAVE TIME - does relatedness predict diversification?


In [ ]:
## The canonical evolutionary-economic-geography test: regions enter new
## technologies that are related to what they already do. We have two periods.
d0 <- reg_tech[period == "2010-2014" & nuts2 %in% rownames(X) & cpc4 %in% colnames(X)]
X0 <- matrix(0, nrow(X), ncol(X), dimnames = dimnames(X))
X0[cbind(d0$nuts2, d0$cpc4)] <- d0$n_pat
RTA0 <- (X0 / pmax(rowSums(X0), 1)) / rep(colSums(X0) / sum(X0), each = nrow(X0))
M0 <- (RTA0 >= 1) * 1
M0[!is.finite(M0)] <- 0

Co0 <- t(M0) %*% M0
Phi0 <- Co0 / outer(pmax(colSums(M0), 1), pmax(colSums(M0), 1), pmax)
Phi0[!is.finite(Phi0)] <- 0; diag(Phi0) <- 0
dens0 <- (M0 %*% Phi0) / rep(pmax(colSums(Phi0), 1e-9), each = nrow(M0)) * 100

entry <- data.table(
  nuts2 = rep(rownames(M), times = ncol(M)),
  cpc4  = rep(colnames(M), each = nrow(M)),
  had   = as.vector(M0), has = as.vector(M),
  density0 = as.vector(dens0))
entry <- entry[had == 0]                       # only technologies NOT held before
entry[, entered := as.integer(has == 1)]
entry[, mean(entered), by = .(density_bin = cut(density0, breaks = c(-1, 5, 10, 20, 100)))][order(density_bin)]

summary(glm(entered ~ density0, data = entry, family = binomial))$coefficients
## => the probability of entering a new green technology increases with the
##    relatedness density of that technology to the regional portfolio: the
##    network is the measurement device behind the "principle of relatedness".

---

## Exercises and the five-question checklist


In [ ]:

## ---------------------------------------------------------------------------
## 1. HOW FRAGILE ARE THE RANKINGS? (patents)
## ---------------------------------------------------------------------------
## Rebuild the green co-invention network keeping only patents with at most 10
## inventors, and compare the top-10 inventors by betweenness with the full
## network. How many names survive?
## Hint: proj_two_mode(inv, "appln_id", "person_id", max_size = 10)

## ---------------------------------------------------------------------------
## 2. WHO BRIDGES GREEN AND NON-GREEN TECHNOLOGY? (patents)
## ---------------------------------------------------------------------------
## Using pat_all_inventors_ITgreen.csv.gz, classify each inventor as green-only,
## non-green-only or mixed, and test whether "mixed" inventors have higher
## betweenness in the overall network. This is the empirical core of several
## papers on green technology recombination.
## Hint: inv_all[, .(green_share = mean(green)), by = person_id] then merge on
## the vertex table and compare distributions.

## ---------------------------------------------------------------------------
## 3. A NATIONAL SUBNETWORK (CORDIS)
## ---------------------------------------------------------------------------
## Take the Horizon Europe organisation network, extract the subgraph of Italian
## organisations, and find (a) the most central ones, (b) the share of their ties
## that stay inside Italy. Repeat for another country and compare.
## Hint: induced_subgraph(g, V(g)[country == "IT"]) for (a); for (b) work on the
## full edge list and use the country attribute of both endpoints.

## ---------------------------------------------------------------------------
## 4. MONEY AND POSITION (CORDIS)
## ---------------------------------------------------------------------------
## Aggregate the organisation network at country level and check whether
## betweenness in the country network is correlated with the EC contribution
## received per participation. Who punches above its weight?

## ---------------------------------------------------------------------------
## 5. YOUR OWN LITERATURE (OpenAlex)
## ---------------------------------------------------------------------------
## Change the query in 03_publications.R to the topic of your PhD, rebuild the
## institution network and identify the 10 most central institutions. Then look
## at them by hand: do you recognise duplicates or aggregation problems?

## ---------------------------------------------------------------------------
## 6. PERSISTENCE OF REGIONAL KNOWLEDGE STRUCTURES (indicators)
## ---------------------------------------------------------------------------
## Recompute variety, coherence and complexity for the period 2010-2014 and
## correlate them with the 2015-2019 values. Which indicator is most persistent?
## Then regress the growth of green patents 2015-2019 on the 2010-2014
## indicators. (Careful: this is a descriptive exercise, not a causal claim.)

## ---------------------------------------------------------------------------
## 7. STAY IN TWO MODES (advanced)
## ---------------------------------------------------------------------------
## Everything we did projected the two-mode network into one mode. Try instead
## to work directly on the bipartite graph: build it with the incidence matrix
## returned by proj_two_mode() (element $incidence), set the "type" attribute,
## and compute bipartite degree and clustering.
## Hint:
##   B  <- pr$incidence
##   gb <- graph_from_biadjacency_matrix(B)
##   table(V(gb)$type)
## Which measures still make sense? Which ones do not?

## ---------------------------------------------------------------------------
## CHECKLIST - the five questions to ask before you trust a network result
## ---------------------------------------------------------------------------
## 1. NODES     - are the actor identifiers disambiguated? (inventor names,
##                organisation ids, author ids). Errors in nodes are not noise:
##                they systematically split hubs and destroy paths.
## 2. TIES      - what does the tie mean? co-participation is not interaction.
##                Are mega-events (200-partner projects, 68-inventor patents)
##                creating cliques that drive your topology?
## 3. BOUNDARY  - which actors/events are in the population, and why? Country,
##                technology, sector and time-window choices all move the result.
## 4. TIME      - is the network a snapshot, a cumulative window, or a moving
##                window? Centrality is not comparable across window lengths.
## 5. INFERENCE - are you describing or estimating? Network measures are
##                generated regressors: they are endogenous to the same process
##                you are explaining, and they are correlated across units by
##                construction (no independence).